# QFF 2026 — PS-O1: Weighted Max-Cut QAOA on Processor A vs B

**Reproducible benchmark (participant-defined processors).** This notebook uses the same four-logical-qubit problem on a five-physical-qubit Processor A and nine-physical-qubit Processor B. The noise and coupling definitions are assumptions documented in `processors/`. Run cells from top to bottom from the repository root.

**Crucial:** Classical measurement bits are attached to logical qubits *before* transpilation; the transpiler preserves classical measurement semantics even if it routes the qubits. We do not interpret unmeasured physical-qubit bitstrings as logical solutions.

In [ ]:
# Run this notebook from the repository root, with src/ and processors/ beside it.
# In Colab, first upload/extract the complete repository and set os.chdir(project_path).
# Install dependencies with: %pip install -q -r requirements.txt
import sys, subprocess
try:
    import qiskit, qiskit_aer
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from src.problem import EDGES, N, score, optimum, qaoa, measure
from src.processors import load
from src.routing import compile_circuit
from src.metrics import resources, summarize_counts
from src.visualization import bar_comparison
import qiskit
import qiskit_aer
print('Qiskit',qiskit.__version__,'Aer',qiskit_aer.__version__)

## 1. Input and classical optimum

The weighted graph is participant-defined and held fixed for both architectures. The classical optimum is found by exhaustive enumeration of all 16 assignments.

In [ ]:
import itertools, json
Path('data').mkdir(exist_ok=True)
Path('data/weighted_graph.json').write_text(json.dumps({'num_nodes':N,'weighted_edges':[list(x) for x in EDGES]},indent=2)+'\n')
assignments=list(itertools.product((0,1),repeat=N))
best=max(score(x) for x in assignments)
print('Classical optimum',best,'assignments',[x for x in assignments if score(x)==best])
assert best==12

## 2. Parameter search on the ideal logical workload

We optimise the expectation of the weighted cut under the ideal statevector, not on either processor, ensuring identical parameters for the comparison.

In [ ]:
gammas=np.linspace(0,2*np.pi,41)
betas=np.linspace(0,np.pi/2,31)
score_vector=np.array([score(tuple(int(c) for c in format(i,f'0{N}b')[::-1])) for i in range(2**N)])
best_params=None
for gamma in gammas:
    for beta in betas:
        p=Statevector.from_instruction(qaoa(gamma,beta)).probabilities()
        value=float(np.dot(p,score_vector))
        if best_params is None or value>best_params[0]: best_params=(value,float(gamma),float(beta))
ideal_mean,best_gamma,best_beta=best_params
logical=qaoa(best_gamma,best_beta)
measured=measure(logical)
print('Ideal mean',ideal_mean,'gamma',best_gamma,'beta',best_beta,'ratio',ideal_mean/best)

## 3. Generate processor JSON from source files

These are participant-defined synthetic models. Processor A is a five-node branch; B uses the nine nodes shown in the organiser slide. Both have identical synthetic gate-error rates so that observed differences reflect compiled structure, not differing assumed calibrations.

In [ ]:
from processors.processor_A import write_json as write_A
from processors.processor_B import write_json as write_B
print(write_A(),write_B())
config_A,config_B=load('A'),load('B')
assert config_A['num_qubits']==5 and config_B['num_qubits']==9
print('A edges',config_A['coupling_map'])
print('B edges',config_B['coupling_map'])

## 4. Compile the **same measured** logical circuit on both processors

For a fair first comparison, the initial logical placement is `[0,1,2,3]` for both processors. The Qiskit transpiler may insert routing operations and decomposes the circuit to the declared basis gates. `explicit_swaps` may be zero because routing can be decomposed into CX gates.

In [ ]:
compiled_A=compile_circuit(measured,config_A)
compiled_B=compile_circuit(measured,config_B)
metrics=pd.DataFrame([{'processor':name,'physical_qubits':cfg['num_qubits'],**resources(circ)} for name,cfg,circ in [('A',config_A,compiled_A),('B',config_B,compiled_B)]])
Path('results/tables').mkdir(parents=True,exist_ok=True)
metrics.to_csv('results/tables/processor_comparison.csv',index=False)
display(metrics)

## 5. Exact noiseless correctness test

Because measurements are attached to logical classical bits before compilation, the sampled output keys always represent the same four logical variables. Compare ideal and compiled probability distributions after removing final measurements, marginalising the physical state according to Qiskit's final layout.

In [ ]:
from qiskit.quantum_info import Statevector
ideal=Statevector.from_instruction(logical).probabilities()
for name,circ in [('A',compiled_A),('B',compiled_B)]:
    physical=circ.remove_final_measurements(inplace=False)
    mapping=circ.layout.final_index_layout(filter_ancillas=True)[:N]
    actual=Statevector.from_instruction(physical).probabilities(qargs=mapping)
    tvd=0.5*np.abs(actual-ideal).sum()
    print(name,'logical-to-physical',mapping,'exact TVD',tvd)
    assert tvd<1e-8, 'Noiseless compilation did not preserve the logical distribution'

## 6. Noisy controlled A/B experiment

Ten random seeds, 4096 shots per seed, same uniform depolarizing noise assumptions and the same logical measurement semantics. No hardware-calibrated fidelity claims are made.

In [ ]:
def noise_model(cfg):
    model=NoiseModel()
    p1=cfg['noise']['single_qubit_depolarizing']
    p2=cfg['noise']['two_qubit_depolarizing']
    one=depolarizing_error(p1,1)
    two=depolarizing_error(p2,2)
    for gate in ('rz','sx','x'):
        model.add_all_qubit_quantum_error(one,gate)
    model.add_all_qubit_quantum_error(two,'cx')
    return model

def run_trials(circuit,cfg,seeds=range(100,110),shots=4096):
    sim=AerSimulator(noise_model=noise_model(cfg))
    rows=[]
    for seed in seeds:
        counts=sim.run(circuit,shots=shots,seed_simulator=int(seed)).result().get_counts()
        rows.append({'seed':seed,**summarize_counts(counts)})
    return pd.DataFrame(rows)

raw=pd.concat([run_trials(c,config).assign(processor=name) for name,c,config in [('A',compiled_A,config_A),('B',compiled_B,config_B)]],ignore_index=True)
summary=raw.groupby('processor',as_index=False).agg(mean_cut=('mean_cut','mean'),std_cut=('mean_cut','std'),mean_ratio=('approximation_ratio','mean'))
raw.to_csv('results/tables/noisy_trials.csv',index=False)
summary.to_csv('results/tables/noisy_summary.csv',index=False)
display(summary)

## 7. Layout sensitivity and routing cost

Try valid four-qubit initial placements for both processors. Keep the same ideal logical circuit and noise assumptions. The transpiler may decompose SWAPs into CXs; therefore actual post-decomposition CX count and depth are primary routing resource measures.

In [ ]:
layout_rows=[]
for name,cfg in [('A',config_A),('B',config_B)]:
    candidates=[[0,1,2,3],[1,2,3,4],[0,2,3,4]]
    for layout in candidates:
        circ=compile_circuit(measured,cfg,layout=layout)
        layout_rows.append({'processor':name,'layout':str(layout),**resources(circ)})
layout_df=pd.DataFrame(layout_rows)
layout_df.to_csv('results/tables/layout_comparison.csv',index=False)
display(layout_df)

## 8. Noise sensitivity

Hold the single-qubit error probability fixed at 0.001; vary CX error probability on both processors. Each point averages five 4096-shot trials. This is a controlled sensitivity experiment, not a hardware prediction.

In [ ]:
noise_rows=[]
for name,circ,cfg in [('A',compiled_A,config_A),('B',compiled_B,config_B)]:
    for p in (0,0.005,0.01,0.02,0.03):
        modified=json.loads(json.dumps(cfg))
        modified['noise']['two_qubit_depolarizing']=p
        trials=run_trials(circ,modified,seeds=range(200,205))
        noise_rows.append({'processor':name,'cx_error':p,'mean_cut':trials.mean_cut.mean(),'std_cut':trials.mean_cut.std()})
noise_df=pd.DataFrame(noise_rows)
noise_df.to_csv('results/tables/noise_sensitivity.csv',index=False)
display(noise_df)

## 9. Figures and reproducibility summary

In [ ]:
Path('results/figures').mkdir(parents=True,exist_ok=True)
bar_comparison(metrics,'depth','results/figures/processor_depth.png')
bar_comparison(metrics,'cx_gates','results/figures/processor_cx.png')
bar_comparison(summary,'mean_cut','results/figures/noisy_mean_cut.png')
fig,ax=plt.subplots(figsize=(7,4))
for name,g in noise_df.groupby('processor'):
    ax.errorbar(g.cx_error,g.mean_cut,yerr=g.std_cut,marker='o',label=f'Processor {name}')
ax.set(xlabel='Two-qubit depolarizing error',ylabel='Mean weighted cut',title='Controlled noise sensitivity')
ax.legend();fig.tight_layout();fig.savefig('results/figures/noise_sensitivity.png',dpi=180);plt.close(fig)
print('Figures saved. Classical optimum:',best,'Ideal QAOA mean:',ideal_mean)
print('NOTE: The report must be updated using the actual outputs from this run.')